# 📱 MONARCH STUDIO · phone-only notebook — m-test-01

**Aap ko sirf 3 tap karne hain:**

1. Upar menu → **Runtime** → **Change runtime type** → **T4 GPU** → Save
2. Upar menu → **Runtime** → **Run all**
3. ~8–15 min baad neeche aakhri cell me `DONE` dikhega → mujhe chat me `DONE` likh do

Yeh notebook teeno kaam khud karta hai (jo stage chahiye, neeche toggle se ON/OFF):

| Stage | Kya banega | Free? |
|---|---|---|
| 1 · Clips | 1 silent clips, 480x832, Wan 2.1 1.3B | ✅ $0 |
| 2 · Voice | ~~off in this test~~ | — |
| 3 · Upscale | 480×832 → 960×1664 (Real-ESRGAN ×2) | ✅ $0 |

Sab kuch seedha aap ke Drive folder me jayega:
`Wf-07 • Monarch Ai Agent / 05 · Projects/m-test-01/`

> Mix, verify aur master main yahan se karunga — aap ko ffmpeg ya terminal ki zarurat **nahi**.


In [ ]:
# ── 1 · Setup (koi edit nahi) ────────────────────────────────────────────────
RUN_CLIPS = True       # silent clips (sabse lamba step)
RUN_VOICE = False       # Kokoro voice lines
RUN_FOLEY = False       # MMAudio ambience/SFX
RUN_UPSCALE = True     # 480p -> 2x

import hashlib, json, pathlib, subprocess, time

PROJECT = "m-test-01"
BASE = pathlib.Path(f"/content/drive/MyDrive/MONARCH STUDIO/Wf-07 • Monarch Ai Agent/05 · Projects/{PROJECT}")

from google.colab import drive
drive.mount("/content/drive")

SHOTS = BASE / "shots"
VO_DIR = BASE / "audio" / "vo"
FOLEY_DIR = BASE / "audio" / "foley"
for folder in (SHOTS, VO_DIR, FOLEY_DIR):
    folder.mkdir(parents=True, exist_ok=True)

JOBS = [
  {
    "id": "t01-plate",
    "seed": 42,
    "frames": 77,
    "steps": 30,
    "width": 480,
    "height": 832,
    "prompt": "cinematic medium shot of a single brass desk lamp on a dark walnut table, warm pool of light, soft volumetric haze, slow push in, shallow depth of field, film grain, no people, no text"
  }
]
VO_TEXT = {
  "t01-plate": ""
}
FOLEY_PROMPT = {
  "t01-plate": ""
}
FACTOR = 2

print("project :", BASE, "| exists:", BASE.exists())
print("shots   :", len(JOBS), "| voice:", len([v for v in VO_TEXT.values() if v.strip()]),
      "| foley:", len([v for v in FOLEY_PROMPT.values() if v.strip()]))


def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


In [ ]:
# ── 2 · Wan 2.1 T2V 1.3B load (fp16 + offload; T4 ke liye) ───────────────────
if RUN_CLIPS:
    !pip -q install "diffusers>=0.35" "transformers>=4.49" accelerate imageio imageio-ffmpeg ftfy sentencepiece

    import torch
    from diffusers import AutoencoderKLWan, WanPipeline

    MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
    vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32)
    pipe = WanPipeline.from_pretrained(MODEL_ID, vae=vae, torch_dtype=torch.float16)
    pipe.enable_model_cpu_offload()
    pipe.enable_vae_tiling()
    try:
        pipe.enable_attention_slicing()
    except Exception:
        pass
    print("pipeline ready ·", torch.cuda.get_device_name(0))


In [ ]:
# ── 3 · Clips (silent, 480x832) ─────────────────────────────────────────────
if RUN_CLIPS:
    from diffusers.utils import export_to_video

    NEGATIVE = ("blurry, low quality, distorted, deformed hands, extra limbs, text overlay, "
                "watermark, jpeg artifacts, static frame, flicker")
    receipts = []
    started = time.time()
    for index, job in enumerate(JOBS, 1):
        out = SHOTS / f"{job['id']}.mp4"
        if out.exists() and out.stat().st_size > 10_000:
            print(f"[{index}/{len(JOBS)}] skip (already on Drive) {out.name}")
            continue
        print(f"\n[{index}/{len(JOBS)}] {job['id']}  {job['width']}x{job['height']} · {job['frames']} frames")
        t0 = time.time()
        try:
            frames = pipe(
                prompt=job["prompt"], negative_prompt=NEGATIVE,
                width=job["width"], height=job["height"], num_frames=job["frames"],
                num_inference_steps=job.get("steps", 30), guidance_scale=5.0,
                generator=torch.Generator("cpu").manual_seed(int(job.get("seed", 42))),
            ).frames[0]
            export_to_video(frames, str(out), fps=16)
            print(f"  ✔ {out.name}  {out.stat().st_size/1e6:.1f} MB · {md5(out)[:10]} · {time.time()-t0:.0f}s")
            receipts.append({"id": job["id"], "file": out.name, "md5": md5(out),
                             "seconds": round(time.time() - t0, 1)})
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print("  ✖ OOM — mujhe bolo, main is shot ka size/frames kam kar dunga")
            receipts.append({"id": job["id"], "error": "cuda OOM"})
        except Exception as exc:
            print("  ✖", type(exc).__name__, exc)
            receipts.append({"id": job["id"], "error": f"{type(exc).__name__}: {exc}"})
        (BASE / "colab_clips_receipt.json").write_text(json.dumps(receipts, indent=2))


In [ ]:
# ── 4 · Voice — Kokoro-82M (Apache-2.0, GPU bhi nahi chahiye) ────────────────
if RUN_VOICE:
    !pip -q install kokoro soundfile
    import numpy as np, soundfile as sf
    from kokoro import KPipeline

    pipeline = KPipeline(lang_code="a")      # 'a' American, 'b' British
    VOICE = "af_heart"                       # 54 options; badalna ho to mujhe bolo
    made = 0
    for shot_id, text in VO_TEXT.items():
        text = (text or "").strip()
        if not text:
            continue
        out = VO_DIR / f"{shot_id}.wav"
        if out.exists():
            print("skip (already on Drive)", out.name)
            continue
        chunks = [audio for _, _, audio in pipeline(text, voice=VOICE, speed=1.0)]
        audio = np.concatenate([c if isinstance(c, np.ndarray) else c.numpy() for c in chunks])
        sf.write(out, audio, 24000)
        print("vo", out.name, f"{len(audio)/24000:.2f}s")
        made += 1
    print("voice lines:", made)


In [ ]:
# ── 5 · Foley / ambience — MMAudio (~6 GB, CC-BY-NC weights) ─────────────────
FOLEY_ENGINE = "mmaudio"     # mmaudio | thinksound | off
if RUN_FOLEY and FOLEY_ENGINE == "mmaudio":
    !pip -q install -e git+https://github.com/hkchengrex/MMAudio.git#egg=mmaudio
    !pip -q install torch torchvision torchaudio --upgrade
    import torch, torchaudio
    from mmaudio.eval_utils import all_model_cfg, generate, load_video
    from mmaudio.model.flow_matching import FlowMatching
    from mmaudio.model.networks import get_my_mmaudio
    from mmaudio.model.utils.features_utils import FeaturesUtils

    vram = torch.cuda.get_device_properties(0).total_memory / 2**30
    cfg_name = "mmaudio_small_44k" if vram < 20 else "mmaudio_large_44k_v2"
    model_cfg = all_model_cfg[cfg_name]
    net = get_my_mmaudio(model_cfg.model).eval().to("cuda")
    feats = FeaturesUtils(**model_cfg.features).eval().to("cuda")
    fm = FlowMatching(min_sigma=0, inference_mode="euler", num_steps=25)
    try:
        net.load_weights(torch.load(f"/root/.cache/mmaudio/{cfg_name}.pth", map_location="cpu",
                                    weights_only=True))
    except FileNotFoundError:
        print("weights missing — MMAudio pehli baar khud download karega (demo.py), "
              "ya is cell ko skip karo aur mujhe bolo")
        net = None
    made = 0
    if net is not None:
        for shot_id in JOBS:
            src = SHOTS / f"{shot_id['id']}.mp4"
            out = FOLEY_DIR / f"{shot_id['id']}.wav"
            if not src.exists() or out.exists():
                continue
            info, video = load_video(str(src), 16, model_cfg.sampling_rate)
            with torch.no_grad():
                audio = generate(video, feats, net, fm, num_steps=25,
                                 duration=float(info.duration), cfg_strength=4.5, seed=42)
            torchaudio.save(str(out), audio.cpu(), model_cfg.sampling_rate)
            print("foley", out.name)
            made += 1
    print("foley stems:", made)


In [ ]:
# ── 6 · Upscale 480p → 960x1664 (Real-ESRGAN ×2) ──────────
if RUN_UPSCALE:
    !pip -q install basicsr realesrgan facexlib
    import cv2, shutil, torch
    from basicsr.archs.rrdbnet_arch import RRDBNet
    from realesrgan import RealESRGANer

    net = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
    enhancer = RealESRGANer(
        scale=4, half=True, tile=0, tile_pad=10, pre_pad=0, model=net,
        model_path="https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/RealESRGAN_x4plus.pth")

    def fps_of(src):
        return subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
                               "stream=r_frame_rate", "-of", "csv=p=0", str(src)],
                              capture_output=True, text=True).stdout.strip() or "16/1"

    def upscale_clip(src, dst, factor=FACTOR):
        frames_in, frames_out = pathlib.Path("/content/_in"), pathlib.Path("/content/_out")
        for folder in (frames_in, frames_out):
            shutil.rmtree(folder, ignore_errors=True)
            folder.mkdir(parents=True)
        subprocess.run(["ffmpeg", "-y", "-i", str(src), str(frames_in / "%06d.png")],
                       check=True, capture_output=True)
        for frame in sorted(frames_in.glob("*.png")):
            img = cv2.imread(str(frame), cv2.IMREAD_COLOR)
            big, _ = enhancer.enhance(img, outscale=factor)
            cv2.imwrite(str(frames_out / frame.name), big)
        subprocess.run(["ffmpeg", "-y", "-framerate", fps_of(src),
                        "-i", str(frames_out / "%06d.png"), "-c:v", "libx264", "-crf", "17",
                        "-pix_fmt", "yuv420p", str(dst)], check=True, capture_output=True)
        return dst

    made = 0
    for job in JOBS:
        src = SHOTS / f"{job['id']}.mp4"
        dst = SHOTS / f"{job['id']}-{FACTOR}x.mp4"
        if not src.exists() or dst.exists():
            continue
        print("upscaling", src.name)
        upscale_clip(src, dst)
        print("  ✔", dst.name, f"{dst.stat().st_size/1e6:.1f} MB")
        made += 1
    print("upscaled:", made)


In [ ]:
# ── 7 · Summary + receipt (yeh cell hi "DONE" batata hai) ────────────────────
clips = sorted(SHOTS.glob("*.mp4"))
voice = sorted(VO_DIR.glob("*.wav"))
foley = sorted(FOLEY_DIR.glob("*.wav"))
rows = [{"file": f.name, "mb": round(f.stat().st_size / 1e6, 1), "md5": md5(f)}
        for f in clips + voice + foley]
(BASE / "colab_receipt.json").write_text(json.dumps(
    {"project": PROJECT, "clips": [c.name for c in clips],
     "voice": [v.name for v in voice], "foley": [f.name for f in foley],
     "files": rows}, indent=2))

print("clips :", len(clips))
print("voice :", len(voice))
print("foley :", len(foley))
for row in rows:
    print(f"  {row['file']:<28} {row['mb']:6.1f} MB  {row['md5'][:10]}")

ok = len([c for c in clips if not c.stem.endswith(("-2x", "-4x"))]) >= 1 and len(clips) >= 1
print("\n" + ("DONE ✅  — mujhe chat me 'DONE' likh do, main mix + verify + master bana dunga"
              if ok else "ADHURA ⚠️  — jo missing hai mujhe bolo (main job chhota kar dunga)"))
